# 🏗️ Short Design Report — Multi-Tenant Project Management SaaS trên Microsoft Azure

**Đề tài 06 (Advanced) · Môn Điện toán đám mây · Nhóm 4 thành viên**
Stack: Azure · Java 21 / Spring Boot 3 · PostgreSQL 16 · React (Vite + TS) · Terraform azurerm · GitHub Actions

| Người tổng hợp | Người cung cấp nội dung |
|---|---|
| TV4 | TV1 (hạ tầng, chi phí, CI/CD) · TV2 (API, DB, RLS) · TV3 (Worker, giám sát, k6) |

> **Quy ước:** chỗ ghi `⟦ĐIỀN: …⟧` là **số liệu đo thật**, do thành viên phụ trách cung cấp sau đợt chạy Azure (Tuần 6).
> Không điền số ước đoán vào đó.
>
> **Cập nhật 20/11/2026:** đối chiếu với code và tài liệu thật trên repo (Terraform + `docs/tv1/` của TV1, mã nguồn backend của TV2, nhánh `feat/worker-spring-boot` của TV3). Chỗ nào **thiết kế** khác **hiện trạng code** đều ghi rõ cả hai.

### Mục lục
1. Bài toán · 2. Kiến trúc & ranh giới mạng · 3. Thiết kế Multi-Tenant · 4. Kết quả kiểm chứng bảo mật ·
5. Deployment & Automation · 6. Reliability & Observability · 7. Câu hỏi phản biện · 8. Chi phí & Cleanup · 9. Giới hạn

---
## 1. Bài toán & tiêu chí thành công

Website quản lý dự án/công việc (kiểu Trello/Jira thu nhỏ) phục vụ **nhiều tổ chức (tenant) cùng lúc**. Mỗi tenant có
thành viên, dự án, task board, audit log và export CSV riêng.

> 🔴 **Yêu cầu sống còn:** không rò rỉ dữ liệu giữa các tenant ở **bất kỳ tầng nào** (API, DB, cache, blob, worker, log),
> kể cả khi người dùng cố tình sửa ID trên API hoặc giả mạo job trong hàng đợi.

| Vai trò | Quyền |
|---|---|
| **Owner** | Toàn quyền org: mời/xóa thành viên, đổi vai trò, CRUD dự án/task, export |
| **Member** | CRUD dự án/task, xem lịch sử, export |
| **Guest** | Chỉ đọc. Mọi request ghi bị **API** trả `403` |

---
## 2. Kiến trúc

```text
 Trình duyệt ──HTTPS──► Azure Static Web Apps (React SPA)
     │  OIDC + PKCE
     ├────────────────► Microsoft Entra External ID (CIAM) ──► JWT access token
     │  Bearer JWT, X-Request-Id
     ▼
 App Service (Linux B1, Java 21) ── api profile ──┐          ┌── worker profile (cùng Plan B1)
     │ VNet Integration (snet-app 10.0.1.0/24)    │          │
     ├──► PostgreSQL Flexible B1ms (snet-db 10.0.2.0/24, private DNS, public access = OFF)
     ├──► Storage Account: Queue `export-jobs` (+ `export-jobs-poison`) ──► Worker
     │                     Blob private `exports/{tenant_id}/{job_id}.csv` ◄── Worker
     ├──► Key Vault (RBAC) ◄── Managed Identity (Key Vault reference cho DB password)
     └──► Application Insights / Log Analytics ──► Dashboard 4 biểu đồ, Metric Alert ──► Action Group (Email)
```

⟦ĐIỀN: chèn **Actual Architecture Diagram** của TV1 (ảnh từ Azure Portal / draw.io sau `terraform apply`)⟧

### 2.1 Ranh giới mạng & bảo vệ bí mật

| Thành phần | Biện pháp | Ở đâu trong code |
|---|---|---|
| 🗄️ PostgreSQL | `public_network_access_enabled = false`, subnet delegate, Private DNS Zone. Chỉ App Service (qua VNet Integration) kết nối được | `infra/postgresql.tf`, `infra/network.tf` |
| 📦 Blob | `allow_nested_items_to_be_public = false`, container `private`, TLS ≥ 1.2. **Thiết kế:** tải qua User Delegation SAS read-only ≤ 5 phút, scope 1 blob (Terraform đã cấp role *Storage Blob Delegator* cho API). **Hiện trạng code TV2:** service SAS ký bằng account key qua connection string, hạn **10 phút**, cho phép **HTTPS và HTTP**, chưa có endpoint tải | `infra/storage_queue.tf`, `infra/app_service.tf`, `BlobSasService.java` |
| 🔑 Secrets | Managed Identity; DB password chỉ có trong Key Vault (`@Microsoft.KeyVault(SecretUri=…)`); Storage dùng RBAC, không account key; SPA là public client (PKCE), không client secret | `infra/keyvault.tf`, `infra/app_service.tf` |
| 🔒 HTTPS | `https_only = true`; SWA mặc định HTTPS; CSP, HSTS, `X-Frame-Options: DENY`, `Referrer-Policy: no-referrer` (để SAS không lọt qua Referer) | `frontend/public/staticwebapp.config.json` |

Cell dưới đây **đọc trực tiếp Terraform** để kiểm tra các khẳng định trên còn đúng với code hiện tại:

In [1]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists() or (p / "infra").is_dir())
print("Repo root:", ROOT.name)

Repo root: multi-tenant-project-management


In [2]:
import re
infra = {p.name: p.read_text(encoding="utf-8") for p in (ROOT / "infra").glob("*.tf")}
checks = [
    ("DB tắt public access",         "postgresql.tf",    r"public_network_access_enabled\s*=\s*false"),
    ("Blob cấm anonymous",           "storage_queue.tf", r"allow_nested_items_to_be_public\s*=\s*false"),
    ("Container exports private",    "storage_queue.tf", r'container_access_type\s*=\s*"private"'),
    ("TLS tối thiểu 1.2",            "storage_queue.tf", r'min_tls_version\s*=\s*"TLS1_2"'),
    ("App Service HTTPS only",       "app_service.tf",   r"https_only\s*=\s*true"),
    ("DB password qua Key Vault ref","app_service.tf",   r"@Microsoft\.KeyVault"),
    ("Managed Identity",             "app_service.tf",   r'type\s*=\s*"SystemAssigned"'),
    ("Storage tắt public network",   "storage_queue.tf", r"public_network_access_enabled\s*=\s*false"),
    ("Có Static Web App",            "*",                r'resource\s+"azurerm_static_web_app"'),
]
for label, f, rx in checks:
    text = "\n".join(infra.values()) if f == "*" else infra.get(f, "")
    print(("✅" if re.search(rx, text) else "❌ CHƯA CÓ"), label)

✅ DB tắt public access
✅ Blob cấm anonymous
✅ Container exports private
✅ TLS tối thiểu 1.2
✅ App Service HTTPS only
✅ DB password qua Key Vault ref
✅ Managed Identity
❌ CHƯA CÓ Storage tắt public network
✅ Có Static Web App


---
## 3. Thiết kế Multi-Tenant (phòng thủ nhiều lớp)

Mô hình **shared database, shared schema**: mọi bảng nghiệp vụ có `tenant_id`. Đây là lựa chọn rẻ nhất cho B1ms, đổi lại
isolation phải được bảo đảm bằng **nhiều lớp độc lập**, để một lớp có lỗi thì lớp sau vẫn chặn được.

| # | Lớp | Cơ chế | Chặn được |
|---|---|---|---|
| 1 | Xác thực | Spring Security verify JWT Entra (issuer, audience, chữ ký, hạn) | Không token / token giả `alg=none` → `401` |
| 2 | Ủy quyền theo tenant | `TenantMembershipInterceptor`: đọc `tenantId` từ path, tra **memberships mỗi request**, kiểm role | Org không thuộc về mình → `403`; Guest ghi → `403`; thu hồi có hiệu lực **ngay** |
| 3 | Truy vấn có phạm vi | `WHERE tenant_id = :verifiedTenant AND id = :id` | IDOR đổi ID → `404` |
| 4 | PostgreSQL RLS | `SET LOCAL app.tenant_id` + policy `tenant_id = current_setting('app.tenant_id')::uuid` | Code quên điều kiện tenant vẫn bị DB chặn |
| 5 | Ràng buộc dữ liệu | `UNIQUE(tenant_id, name)` trên `projects` | Hai tenant trùng tên không xung đột (Test 3) |
| 6 | Cache | Khóa `tenant:{id}:…`, invalidate theo tenant | Lẫn dữ liệu qua cache (Test 3) |
| 7 | Blob | Path `exports/{tenant_id}/{job_id}.csv` lấy từ DB, SAS scope 1 blob | Đoán/tải file tenant khác |
| 8 | Worker | Message queue **không đáng tin**: đối chiếu `claimedTenantId` với `export_jobs.tenant_id`, kiểm lại membership | Job giả mạo → từ chối, log `jobId`, metric, Alert |
| 9 | Log | JSON log + MDC `requestId/jobId/tenantId`; redact JWT/password/SAS | Rò rỉ bí mật qua log |

**Hiện trạng code TV2** (`V2__enable_rls.sql`): bật RLS trên `tenant_memberships`, `workspaces`, `audit_events`, `export_jobs` với policy:
```sql
CREATE POLICY workspaces_isolation ON workspaces
  USING      (tenant_id = NULLIF(current_setting('app.tenant_id', true), '')::uuid)
  WITH CHECK (tenant_id = NULLIF(current_setting('app.tenant_id', true), '')::uuid);
```
Mỗi request gọi `select set_config('app.tenant_id', ?, true)` trong transaction (tương đương `SET LOCAL`), nên tenant context
tự mất khi transaction kết thúc và không rò sang request sau trên cùng connection HikariCP. Thiếu context thì policy so với `NULL` và không trả hàng nào.

Còn mở: chưa dùng `FORCE ROW LEVEL SECURITY`, chưa tạo role DB runtime không sở hữu bảng (TV2 tự ghi trong `IMPLEMENTATION_STATUS.md`).
Nếu app chạy bằng tài khoản sở hữu bảng thì RLS **không có hiệu lực**. ⟦ĐIỀN (TV2): role runtime thật khi deploy⟧

### 3.1 Mô hình dữ liệu (6 bảng)

**Hiện trạng code TV2** (`V1__create_tables.sql`):

| Bảng | Cột chính | Ghi chú |
|---|---|---|
| `tenants` | `id, name, created_at, updated_at` | |
| `users` | `id, entra_object_id (UNIQUE), email, display_name` | Ánh xạ danh tính Entra (`oid`, nếu không có thì `sub`) |
| `tenant_memberships` | `tenant_id, user_id, role, status` | `role` ∈ OWNER/MEMBER/GUEST; `status` ∈ ACTIVE/INACTIVE; `UNIQUE(tenant_id, user_id)`. Nguồn sự thật cho quyền, kiểm mỗi request |
| `workspaces` | `tenant_id, name, description, created_by` | `UNIQUE(tenant_id, name)` |
| `audit_events` | `tenant_id, actor_user_id, action, resource_type, resource_id, request_id, details JSONB` | Index `(tenant_id, created_at DESC)` |
| `export_jobs` | `tenant_id, requested_by, status, queue_message_id, blob_name, error_message` | `status` ∈ PENDING/PROCESSING/COMPLETED/FAILED. **Nguồn sự thật** cho Worker |

⚠️ **Khác tài liệu định hướng của nhóm:** tài liệu yêu cầu `projects` (có `UNIQUE(tenant_id, name)`) và `tasks`;
code hiện có `workspaces` và **chưa có bảng task**. Seed hiện có 1 tenant và 1 user (tài liệu yêu cầu 3 org × 5 user × 20 task).
⟦ĐIỀN: mô hình cuối cùng sau khi nhóm chốt + ERD⟧

### 3.2 Luồng Export bất đồng bộ
1. `POST /api/v1/tenants/{t}/exports` (đường dẫn hiện tại của TV2) → API ghi `export_jobs(PENDING)` → gửi message vào queue `export-jobs` → `202`.
   - **Thiết kế (parser v1 của TV3):** `{schemaVersion: 1, jobId, claimedTenantId, requestId}`; trường lạ bị từ chối.
   - **Hiện trạng code TV2:** gửi `{jobId, tenantId}`, không có `schemaVersion`, nên Worker sẽ chuyển message vào poison queue. ⟦ĐIỀN: định dạng sau khi TV2 + TV3 chốt⟧
2. Worker nhận message, tra `export_jobs` theo `jobId`, **so tenant**, kiểm membership, claim job nguyên tử.
3. Query tasks bằng tenant **đã xác minh từ DB** → CSV (chống formula injection) → Blob private → `COMPLETED`.
4. UI poll `GET /exports/{jobId}`; xong thì gọi endpoint tải → API cấp SAS read-only → trình duyệt tải thẳng từ Blob.
   (Thiết kế ≤ 5 phút; code hiện tại 10 phút; endpoint tải chưa có trong controller.) CSV của TV3: cột `task_id,title,status`, chống formula injection.

---
## 4. Kết quả kiểm chứng bảo mật

Bộ test tự động `tests/security/` (pytest, gọi thẳng API, ghi raw HTTP + timestamp, tự che token).
Bộ test đã được **tự kiểm chứng** bằng mock API cố tình có lỗ hổng: mock an toàn cho 36/36 PASS, còn mỗi lỗ hổng
(IDOR, membership cũ, unique toàn cục, cache dùng chung, header demo thay JWT) đều làm test tương ứng FAIL. Xem notebook `STATUS_AND_HANDOFF.ipynb`.

| Test | Kịch bản | Kết quả trên Azure | Bằng chứng |
|---|---|---|---|
| Test 1: IDOR | 13 kiểu đọc + 11 kiểu ghi/xóa/export bằng JWT Org A lên ID Org B; token giả; Guest ghi; giả danh bằng header `X-Demo-User` (chế độ demo của backend, mặc định đang bật) | ⟦ĐIỀN⟧ | `docs/tv4/evidence/<run>/test_org_a_cannot_*.http.txt` |
| Test 2: Thu hồi membership | Owner xóa X → dùng lại JWT cũ của X | ⟦ĐIỀN: status = 403, khoảng cách ms⟧ | `…/test_revoked_member_is_blocked_on_next_request.http.txt` |
| Test 3: Trùng tên & cache | A và B cùng tạo "Cloud Project", đọc 2 lần, sửa/xóa ở A | ⟦ĐIỀN⟧ | `…/test_same_project_name_in_two_tenants.http.txt` |
| Failure Drill (TV3) | Message có `claimedTenantId` của org khác | ⟦ĐIỀN: log jobId, metric, email Alert lúc hh:mm, API vẫn 200⟧ | ⟦ĐIỀN: ảnh⟧ |

Cell dưới **tự đọc** bằng chứng đã rà soát trong `docs/tv4/evidence/` (sau Đợt 2 sẽ tự hiện kết quả Azure):

In [3]:
import json
ev_root = ROOT / "docs" / "tv4" / "evidence"
runs = sorted(ev_root.glob("*/summary.json")) if ev_root.exists() else []
if not runs:
    print("⏳ Chưa có bằng chứng Azure trong docs/tv4/evidence/ (sẽ có sau Đợt 2, Tuần 6).")
for s in runs:
    data = json.loads(s.read_text(encoding="utf-8"))
    info = json.loads((s.parent / "run-info.json").read_text(encoding="utf-8"))
    print(f"📁 {s.parent.name}  target={info['target']}  PASS {data['passed']}/{data['passed'] + data['failed']}")
    for group in ("test_1", "test_2", "test_3"):
        rs = [r for r in data["results"] if r["test"].startswith(group)]
        if rs:
            ok = sum(r["outcome"] == "passed" for r in rs)
            print(f"   {group}: {ok}/{len(rs)} PASS")

⏳ Chưa có bằng chứng Azure trong docs/tv4/evidence/ (sẽ có sau Đợt 2, Tuần 6).


---
## 5. Deployment & Automation

| Hạng mục | Cách làm | Trạng thái |
|---|---|---|
| IaC | Toàn bộ hạ tầng trong `infra/` (Terraform azurerm ~> 4.0): RG, VNet 2 subnet, PostgreSQL B1ms private, Storage, Key Vault, App Service B1 (api + worker), Static Web App Free, App Insights, 2 alert. `apply` dựng từ đầu, `destroy` xóa sạch | ✅ Đã có (TV1) |
| CI | `.github/workflows/infra-ci.yml`: chạy khi push `main`/`feat/**` và khi mở PR → `terraform fmt -check`, `init -backend=false`, `validate`; chạy self-test prototype Worker (JDK 21) | ✅ Đã có (TV1) |
| CD | `.github/workflows/deploy-and-rollback.yml` (chạy tay): build JAR bằng Maven, lưu artifact 30 ngày kèm commit SHA, đăng nhập Azure bằng **OIDC Federated Credentials** (không lưu client secret), deploy `api`/`worker`, rồi kiểm tra `/actuator/health` (12 lần × 10 giây) | ✅ Đã có workflow. Mặc định build từ thư mục `backend/`; mã TV2 hiện mới ở dạng file zip. ⟦ĐIỀN: lần deploy thật⟧ |
| Rollback | Gói **B1 không có Deployment Slot** (chỉ có từ S1, ~$70/tháng). Rollback = chạy lại workflow ở chế độ `rollback` với `rollback_run_id` của bản ổn định → deploy lại đúng file JAR cũ (TV1 ước tính ~1 phút) | ⟦ĐIỀN (TV1): thời gian đo thật, link video⟧ |
| Deploy frontend | Static Web App đã có trong Terraform (output `frontend_https_url`); workflow CD chưa có bước build/deploy frontend | ⟦ĐIỀN⟧ |

---
## 6. Reliability & Observability

- Log JSON có `request_id` (API) và `job_id` (Worker). Mỗi lỗi trên UI hiển thị `request_id` để tra ngược.
- **Dashboard 4 biểu đồ** (kế hoạch của TV3, `prototypes/tv3-worker/docs/OBSERVABILITY.md`):
  1. Export job theo kết quả (thành công / lỗi tạm thời / bị từ chối vì bảo mật)
  2. Bộ đếm `worker_tenant_mismatch_total` (tín hiệu cho alert)
  3. p95 thời gian xử lý một export job (khác với p95 HTTP của k6)
  4. Sức khỏe queue: số lần retry, độ dài poison queue / tuổi message cũ nhất

  ⟦ĐIỀN (TV3): ảnh dashboard thật⟧
- **Alert** (Terraform `infra/monitoring.tf`, gửi email qua Action Group):

  | Alert | Điều kiện | Mức |
  |---|---|---|
  | `alert-api-http-5xx` | `Http5xx` của API > 5 trong 5 phút | Sev 2 |
  | `alert-worker-forged-tenant-mismatch` | `worker_tenant_mismatch_total` > 0 trong 5 phút (`skip_metric_validation`) | Sev 1 |

  Hiện trạng: code Worker mới có interface `SecurityMetrics`, **chưa phát metric này** lên App Insights.
  Thời gian từ lúc phát hiện tới lúc email đến phải **đo thật**, không mặc định 1–2 phút. ⟦ĐIỀN (TV3): mốc thời gian + ảnh email⟧
- **Load test k6** (`prototypes/tv3-worker/k6/api-load.js`): 15 giây warmup → 2 phút giữ 5 VU → 15 giây cooldown;
  ngưỡng `http_req_failed < 1%`, `p95 < 800 ms` (mục tiêu gợi ý, không phải yêu cầu đề). ⟦ĐIỀN (TV3): p95 = … ms, error rate = … %⟧

Truy vết một request từ file bằng chứng sang App Insights:
```kusto
union traces, requests
| where customDimensions.request_id == "tv4-A-owner-..."
| order by timestamp asc
```

---
## 7. Câu hỏi phản biện

### 7.1 Scaling: chuyện gì xảy ra khi tải tăng 10×?

| Thành phần | Hành vi khi scale | Hướng xử lý |
|---|---|---|
| API | **Stateless** (JWT + membership trong DB) nên scale ngang ngay | Cache in-process → Azure Cache for Redis (giữ khóa `tenant:{id}:…`) khi nhiều instance |
| Worker | Scale độc lập; Queue hấp thụ đột biến, API không bị chậm | Tách Plan / Container Apps scale theo độ dài queue |
| **PostgreSQL B1ms** | **Nút cổ chai đầu tiên** (1 vCPU burstable, ít kết nối) | Index `(tenant_id, …)`, phân trang → General Purpose → read replica → PgBouncer |
| Noisy neighbor | 1 tenant export lớn làm nghẽn tenant khác | Giới hạn job đồng thời/tenant, rate limit theo tenant, stream CSV |
| Tenant doanh nghiệp | Cần tách biệt vật lý | DB riêng (silo), lớp 2–3 giữ nguyên, thêm định tuyến connection |

### 7.2 Connection Pool

**Ràng buộc:** `số instance × (pool API + pool Worker) + dự trữ hệ thống ≤ max_connections`

- Pool nhỏ + transaction ngắn tốt hơn pool lớn: 1 vCPU không chạy song song nhiều truy vấn hơn số core.
- ⚠️ **Rủi ro riêng của multi-tenant:** connection dùng lại có thể mang `app.tenant_id` của request trước.
  Biện pháp: `SET LOCAL` (chỉ sống trong transaction), hoặc reset khi trả về pool.
  Test 1 và 3 xen kẽ hai tenant trên cùng pool để phát hiện lỗi này.
- Scale-out nhân số kết nối lên → giảm pool/instance hoặc bật PgBouncer.

**Số liệu hiện có:** code TV2 để `maximum-pool-size = ${DB_POOL_MAX:10}`, `minimum-idle = 2`. Tài liệu TV1 ghi B1ms có `max_connections` khoảng 50 và đề xuất giảm pool xuống **5** mỗi instance (hoặc bật PgBouncer có sẵn của Flexible Server). ⟦ĐIỀN: kết quả `SHOW max_connections;` trên server thật⟧

▶️ Đổi tham số trong cell dưới để xem cấu hình nào an toàn khi scale:

In [4]:
# Tham số — TV2 cập nhật bằng số thật (max_connections lấy từ: SHOW max_connections;)
MAX_CONNECTIONS = 50      # giá trị mặc định tham khảo cho Burstable B1ms — cần xác nhận trên server thật
RESERVED        = 5       # dự trữ cho admin/hệ thống Azure, migration, psql khi debug
CONFIGS = {   # (pool mỗi instance API, pool mỗi instance Worker)
    "Hiện tại (mặc định code TV2: 10)": (10, 10),
    "Đề xuất của TV1 (5)":              (5, 5),
}
limit = MAX_CONNECTIONS - RESERVED
for name, (api_pool, worker_pool) in CONFIGS.items():
    print()
    print(f"▶ {name} — giới hạn dùng được: {limit}")
    print(f"{'API inst':>8} {'Worker inst':>11} {'Tổng kết nối':>13}  Kết luận")
    for api_n, worker_n in [(1, 1), (2, 1), (3, 1), (4, 2), (6, 2)]:
        total = api_n * api_pool + worker_n * worker_pool
        verdict = "✅ an toàn" if total <= limit else f"❌ vượt {total - limit} → giảm pool hoặc dùng PgBouncer"
        print(f"{api_n:>8} {worker_n:>11} {total:>13}  {verdict}")


▶ Hiện tại (mặc định code TV2: 10) — giới hạn dùng được: 45
API inst Worker inst  Tổng kết nối  Kết luận
       1           1            20  ✅ an toàn
       2           1            30  ✅ an toàn
       3           1            40  ✅ an toàn
       4           2            60  ❌ vượt 15 → giảm pool hoặc dùng PgBouncer
       6           2            80  ❌ vượt 35 → giảm pool hoặc dùng PgBouncer

▶ Đề xuất của TV1 (5) — giới hạn dùng được: 45
API inst Worker inst  Tổng kết nối  Kết luận
       1           1            10  ✅ an toàn
       2           1            15  ✅ an toàn
       3           1            20  ✅ an toàn
       4           2            30  ✅ an toàn
       6           2            40  ✅ an toàn


### 7.3 Retry Safety / Idempotency

Azure Queue giao hàng **at-least-once**: message có thể đến 2 lần, hoặc đến sau khi Worker chết giữa chừng.

| Tình huống | Vì sao vẫn an toàn |
|---|---|
| Message đến 2 lần | Claim nguyên tử: `UPDATE export_jobs … WHERE status = 'PENDING' (hoặc lease hết hạn)`; job đã `COMPLETED` thì bỏ qua (`ALREADY_COMPLETED`) |
| Worker chết khi đang ghi blob | Path blob xác định theo `jobId`: ghi lại thì ghi đè đúng file, không sinh rác |
| Worker chết trước khi commit | Chỉ xóa message **sau** khi commit kết quả cuối → message hiện lại sau visibility timeout |
| Lỗi tạm thời (mạng, throttling) | Backoff + jitter; quá số lần thì vào `export-jobs-poison` với metadata đã làm sạch |
| Người dùng bấm Export 2 lần | Tạo 2 job độc lập, vô hại; có thể thêm header `Idempotency-Key` |

**Cấu hình thật của Worker TV3** (`application-worker.yml`): visibility timeout 30 giây, tự gia hạn mỗi 15 giây khi đang xử lý;
tối đa 5 lần nhận; retry backoff lũy thừa 5 giây → tối đa 5 phút, có jitter. Message sang poison queue chỉ mang
`sourceMessageId, errorCode, dequeueCount, failedAt`, **không chép nội dung gốc**. Message chỉ bị xóa sau kết quả bền vững
(`DURABLY_COMPLETED` / `DURABLY_REJECTED`), và luôn dùng `popReceipt` mới nhất.

Bằng chứng hiện có: TV3 báo 16 unit test pass (chạy trên JDK 25, chưa thử Azurite/Azure) và 5/5 self-test prototype.
Phần xử lý job thật (`ExportMessageProcessor`) chưa có, đang chờ schema của TV2.
Chi tiết: `prototypes/tv3-worker/docs/CONTRACT_TO_CONFIRM.md`. ⟦ĐIỀN (TV3): kết quả E2E retry/poison⟧

### 7.4 Alternative Compute

| Lựa chọn | Ưu | Nhược | Kết luận |
|---|---|---|---|
| **App Service B1 (đã chọn)** | HTTPS sẵn, VNet Integration, Managed Identity + Key Vault reference, Java 21, chi phí cố định thấp, API + Worker chung 1 Plan, không cần Load Balancer/NAT riêng | Không scale về 0; **không có Deployment Slot** (rollback bằng redeploy artifact) | ✅ Phù hợp nhất với $100 và chiến lược bật theo đợt |
| Azure Container Apps | Scale về 0, scale Worker theo queue (KEDA), revision để rollback | Cold start Spring Boot; gắn VNet riêng phức tạp hơn và tốn thêm phí hạ tầng mạng | Ứng viên tốt nhất cho Worker khi tải thật |
| Azure Functions (Queue trigger) | Trả theo lần chạy, tự scale theo queue | Spring Boot nặng, giới hạn thời gian chạy, VNet cần Premium/Flex | Hợp với Worker nhỏ viết lại |
| AKS | Linh hoạt tối đa | Chi phí & vận hành vượt quy mô đồ án | ❌ Không chọn |

---
## 8. Chi phí & Cleanup

Ước tính của TV1 (`docs/tv1/COST_ESTIMATE_AND_CLEANUP.md`). Workload giả định: 3 org × 5 user × 20 task, ~150.000 request/tháng,
~1.000 export job/tháng, DB 32 GiB + backup 7 ngày, log < 1 GiB/tháng, egress < 5 GiB/tháng.
Region Southeast Asia; riêng Static Web Apps đặt control-plane ở East Asia.

| Tài nguyên | SKU | Bật 24/7 (USD/tháng) | Một đợt test (~15 giờ) |
|---|---|---:|---:|
| App Service Plan (api + worker chung) | B1 Linux | ~13.14 | ~0.27 |
| PostgreSQL Flexible Server | B_Standard_B1ms + 32 GiB | ~16.06 | ~0.50 |
| Storage Account (Blob + Queue) | Standard LRS | ~0.05 | ~0.01 |
| Key Vault | Standard | ~0.03 | ~0.01 |
| VNet + Private DNS Zone | | ~0.50 | ~0.02 |
| Log Analytics, App Insights, 2 Metric Alert | PerGB2018 | ~0.20 | ~0.01 |
| Static Web Apps, Entra External ID, egress < 5 GB | Free | 0 | 0 |
| **Tổng** | | **~29.98** | **~0.82** |

⟦ĐIỀN: chi phí thực tế từ Azure Cost Management sau các đợt⟧

**Kiểm soát:** Cost Alert ở $25 / $50 / $75. Đây chỉ là email thông báo, **không tự tắt** tài nguyên. Giữ cố định 1 instance B1,
không bật autoscale; k6 chỉ chạy 2,5–3 phút. 85% thời gian dev chạy local (Docker PostgreSQL + Azurite, hoặc mock), chi phí $0.

**Cleanup** (TV1 thực hiện sau mỗi đợt):
```powershell
cd infra
terraform destroy -auto-approve
az group list --query "[?starts_with(name, 'rg-mtpm-saas')].{Name:name}" -o table   # phải trống
az resource list -o table
az postgres flexible-server list -o table
az appservice plan list -o table
az keyvault list-deleted -o table   # Key Vault đã purge (purge_soft_delete_on_destroy = true)
```
⟦ĐIỀN: ảnh Resource Group trống sau đợt cuối⟧

---
## 9. Giới hạn & hướng phát triển
- Cache in-process chưa dùng chung giữa nhiều instance (xem 7.1).
- Mời thành viên đang thêm trực tiếp theo email; production cần lời mời có xác nhận và hết hạn.
- Chưa có WAF/Front Door hay rate limit theo tenant.
- **Rủi ro tích hợp hiện tại (đọc code ngày 20/11):**
  - Định dạng message queue của API (`{jobId, tenantId}`) chưa khớp parser của Worker (`schemaVersion` bắt buộc, cấm trường lạ).
  - Code API đọc `AZURE_STORAGE_CONNECTION_STRING`, còn Terraform chỉ cấp endpoint + Managed Identity, nên trên Azure API chưa gửi được message và chưa tạo được SAS.
  - Biến cấu hình issuer khác tên (`OIDC_ISSUER_URI` trong Terraform, `ENTRA_ISSUER_URI` trong code), chưa đặt `ENTRA_AUDIENCE`.
  - Chế độ demo (`APP_SECURITY_DEMO_MODE`) mặc định **bật** và Terraform chưa tắt. Nếu deploy nguyên như vậy, header `X-Demo-User` thay được JWT. Test 1 có kiểm tra trường hợp này.
- ⟦ĐIỀN: giới hạn thực tế phát hiện khi chạy Azure⟧